In [ ]:
# Shared project setup for imports and file locations
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FIGURES_DIR = PROJECT_ROOT / 'figures'

def resolve_path(path):
    candidate = Path(path)
    if candidate.exists():
        return candidate
    text = str(path).replace('\\', '/')
    name = Path(text).name
    special = {
        'positive_controls.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'negative_controls.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'Ten_positive_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'Ten_negative_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'fcg.txt': DATA_DIR / 'fcg.txt',
    }
    if name in special:
        return special[name]
    matches = [p for p in PROJECT_ROOT.rglob(name) if '.ipynb_checkpoints' not in p.parts and '.git' not in p.parts]
    if len(matches) == 1:
        return matches[0]
    if (text.startswith('/Users/') or text.startswith('/home/') or ':\\' in text) and '.' not in name:
        return PROJECT_ROOT
    return candidate

from pdm_learn.preprocessing import build_density_map, density_centers, densitymap, drop_nan, extract, mut_trim, normalize, trim, trim_pairs
from pdm_learn.modeling import KFold_PR, LOOCV, LOOCV_grouped_plot, area_table, core_predict, heatmap, importance_test, ks_pvalue
from pdm_learn.simulation import eps, partition


In [2]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from bisect import bisect_left
import pickle

In [3]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [4]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [5]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [6]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [7]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [8]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [9]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [10]:
gene_exp = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'Gene_Expression_Trimmed.csv')
gene_exp.name = 'gene_exp'

copy_num = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'Copy_Number_Trimmed.csv')
copy_num.name = 'copy_num'

shRNA = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'shRNA_Trimmed.csv')
shRNA.name = 'shRNA'

gene_mut = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'Gene_Mutation_Trimmed.csv')
gene_mut.name = 'gene_mut'

CRISPR = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'CRISPR_Trimmed.csv')
CRISPR.name = 'CRISPR'

In [11]:
pos = pd.read_pickle(resolve_path('/Users/jzx/Documents/Computer Science/CCLE_ML_project/Nov24_update_required_files/Ten_positive_controls_1119.pkl'))
neg = pd.read_pickle(resolve_path('/Users/jzx/Documents/Computer Science/CCLE_ML_project/Nov24_update_required_files/Ten_negative_controls_1119.pkl'))

In [12]:
genes = sorted(set(gene_exp.iloc[:,0].str.strip()) & 
               set(copy_num.iloc[:,0].str.strip()) & 
               set(shRNA.iloc[:,0].str.strip()) & 
               set(CRISPR.iloc[:,0].str.strip()))
genes = [i.strip() for i in genes]

In [13]:
datasets = gene_exp, copy_num, shRNA, gene_mut, CRISPR
points = density_centers(gene_exp, 7), [0,1,2,3,4,6,8], density_centers(shRNA, 7), [0, 1], density_centers(CRISPR, 7)
cont = True, False, True, False, True

In [14]:
temp_time = time.time()
pos_out = []
neg_out = []
iterations = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']
for i in iterations:
    pos_out.append(build_density_map(datasets, [pair for group in list(pos['positive_control_' + i].values()) for pair in group], points, cont))
    neg_out.append(build_density_map(datasets, neg['negative_controls_' + i], points, cont))
print(time.time()-temp_time)

ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1
ANAPC3
ANAPC9
ANAPC8
MRE11
ATRIP
STIP1


/var/folders/29/4xb238qn6yz1n58ysdvl_gsm0000gn/T/ipykernel_59628/139204224.py:45: RuntimeWarning: invalid value encountered in divide
  mat /= len(x)


ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1
ANAPC6
ANAPC3
MRE11
ATG13
STIP1


/var/folders/29/4xb238qn6yz1n58ysdvl_gsm0000gn/T/ipykernel_59628/139204224.py:45: RuntimeWarning: invalid value encountered in divide
  mat /= len(x)


ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11
ATG13
STIP1
ANAPC3
ANAPC6
ANAPC8
ANAPC9
MRE11


...

ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1
ANAPC6
ANAPC8
ANAPC9
ATRIP
ATG13
STIP1


/var/folders/29/4xb238qn6yz1n58ysdvl_gsm0000gn/T/ipykernel_59628/139204224.py:45: RuntimeWarning: invalid value encountered in divide
  mat /= len(x)


ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1
ANAPC3
ANAPC8
ANAPC6
ATRIP
STIP1


/var/folders/29/4xb238qn6yz1n58ysdvl_gsm0000gn/T/ipykernel_59628/139204224.py:45: RuntimeWarning: invalid value encountered in divide
  mat /= len(x)


18821.036331892014


In [37]:
# add complex names for each pair
for i in range(len(pos_out)):
    complex_names = []
    d = pos['positive_control_' + str(i + 1)]
    for key in d.keys():
        for _ in range(len(d[key])):
            complex_names.append(key)
    pos_out[i] = pd.concat([pd.DataFrame({'Complex':complex_names}), pos_out[i]], axis = 1)

In [42]:
with open(ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl', 'wb') as file:
    pickle.dump(pos_out, file)

In [41]:
pos_out[5]

,Complex,pair,gene_exp.gene_exp.0,gene_exp.gene_exp.1,gene_exp.gene_exp.2,gene_exp.gene_exp.3,gene_exp.gene_exp.4,gene_exp.gene_exp.5,gene_exp.gene_exp.6,gene_exp.gene_exp.7,...,CRISPR.CRISPR.39,CRISPR.CRISPR.40,CRISPR.CRISPR.41,CRISPR.CRISPR.42,CRISPR.CRISPR.43,CRISPR.CRISPR.44,CRISPR.CRISPR.45,CRISPR.CRISPR.46,CRISPR.CRISPR.47,CRISPR.CRISPR.48
0,Cyclin–CDK complexes,CDK4.CCND1,-7.022073,-6.407506,-4.464403,-3.197530,-4.790933,-7.108561,-7.174689,-7.102966,...,-2.457650,-3.534667,-5.738818,-6.267278,-5.702619,-5.325112,-4.141204,-3.550542,-4.493729,-6.406544
1,Cyclin–CDK complexes,CCNE1.CCNA2,-7.174693,-7.166735,-7.103614,-7.132774,-7.172948,-7.174714,-7.174724,-7.172061,...,-4.802601,-6.196282,-6.666088,-6.671337,-6.663114,-6.466731,-6.149890,-6.498971,-6.659328,-6.671980
2,Cyclin–CDK complexes,CDK1.CDK2,-7.173736,-7.145351,-7.128191,-7.164030,-7.174484,-7.174723,-7.174724,-7.106428,...,-4.447469,-6.219067,-6.196561,-6.524807,-6.573285,-6.176364,-5.915626,-6.470899,-6.665007,-6.655805
3,Cyclin–CDK complexes,CDK6.CCNB1,-7.174521,-7.144927,-6.970881,-7.029406,-7.126750,-7.173511,-7.174723,-7.165653,...,-4.754582,-5.257541,-6.310716,-6.646780,-6.575397,-6.030350,-6.265245,-5.969172,-6.401223,-6.662769
4,RB–E2F repressor complex,E2F1.E2F3,-7.174699,-7.173618,-7.172643,-7.173774,-7.174650,-7.174724,-7.174724,-7.166883,...,-4.494993,-5.364487,-6.475017,-6.671827,-6.652730,-6.395005,-6.502056,-6.404336,-6.400576,-6.657118
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,Proteasome complex,PSMD13.PSMD14,-7.174724,-7.174373,-7.167501,-7.171327,-7.174653,-7.174724,-7.174724,-7.174655,...,-4.752290,-6.359419,-6.670323,-6.644979,-6.154521,-6.385175,-6.404602,-6.355734,-6.636815,-6.671940
71,HSP90–CDC37 chaperone complex,HSP90AA1.HSP90AB1,-7.161081,-7.083710,-7.101561,-7.165129,-7.174612,-7.174724,-7.174724,-7.064462,...,-5.361926,-6.525087,-6.671358,-6.672010,-6.658070,-6.455324,-6.398999,-6.637158,-6.670155,-6.672026
72,HSP90–CDC37 chaperone complex,STIP1.CDC37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
73,ISR/EIF2α–ATF4 complex,EIF2S1.ATF4,-7.174693,-7.174634,-7.174402,-7.174019,-7.174551,-7.174721,-7.174724,-7.151553,...,-4.943751,-6.167011,-6.645644,-6.663926,-6.642712,-6.609219,-6.567405,-6.641253,-6.666817,-6.671898
